# ML — Predicción del Precio de Coches de Segunda Mano en Europa

**Objetivo:** Construir un modelo de Machine Learning capaz de predecir el precio de un coche de segunda mano en el mercado español, a partir de un dataset de anuncios reales de AutoScout24 con datos de varios países europeos.

**Dataset:** EU Used Cars — Kaggle (alemazz11/cars-europe)  
**Mercado objetivo:** España  
**Tipo de problema:** Regresión supervisada

In [71]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

## Paso 1: Obtención de datos

In [72]:
path = kagglehub.dataset_download("alemazz11/cars-europe")
print(path)

/home/jorge/.cache/kagglehub/datasets/alemazz11/cars-europe/versions/1


In [73]:
df = pd.read_csv(path + "/fullGas.csv")
df.head()

,Make,Model,Body,Mileage_km,Price,Year,Country,Condition,Fuel_Type,Fuel_Consumption_l,Drivetrain,Gearbox,Gears,Power_hp,Engine_Size_cc,Cylinders,Seats,Doors,Color,Upholstery,Full_Service_History,Non_Smoker_Vehicle,Previous_Owners,Seller,Image_url
0,Abarth,595,Compact,98000,16900,2020.0,IT,Used,Gasoline,6.7,Front Wheel Drive,Automatic,NaN,179,1368.0,4.0,4.0,3.0,White,Full leather,False,False,NaN,Dealer,https://prod.pictures.autoscout24.net/listing-...
1,Abarth,595,Sedan,91500,12500,2017.0,IT,Used,Gasoline,6.0,Front Wheel Drive,Manual,5.0,165,1368.0,4.0,4.0,3.0,Grey,Full leather,False,True,4.0,Dealer,https://prod.pictures.autoscout24.net/listing-...
2,Abarth,595,Compact,40000,17990,2015.0,IT,Used,Gasoline,6.5,Front Wheel Drive,Manual,5.0,300,1368.0,4.0,4.0,3.0,Bronze,Full leather,True,True,1.0,Dealer,https://prod.pictures.autoscout24.net/listing-...
3,Abarth,500,Compact,133000,9300,2008.0,IT,Used,Gasoline,6.5,Front Wheel Drive,Manual,5.0,160,1368.0,4.0,4.0,3.0,Black,Cloth,True,True,2.0,Dealer,https://prod.pictures.autoscout24.net/listing-...
4,Abarth,595,Compact,61019,14990,2021.0,BE,Used,Gasoline,NaN,Front Wheel Drive,Manual,6.0,145,1368.0,4.0,4.0,3.0,Yellow,Part leather,True,True,1.0,Dealer,https://prod.pictures.autoscout24.net/listing-...


In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40494 entries, 0 to 40493
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Make                  40494 non-null  str    
 1   Model                 40273 non-null  str    
 2   Body                  40493 non-null  str    
 3   Mileage_km            40494 non-null  int64  
 4   Price                 40494 non-null  int64  
 5   Year                  38799 non-null  float64
 6   Country               37902 non-null  str    
 7   Condition             40494 non-null  str    
 8   Fuel_Type             40465 non-null  str    
 9   Fuel_Consumption_l    12773 non-null  float64
 10  Drivetrain            31860 non-null  str    
 11  Gearbox               40014 non-null  str    
 12  Gears                 25115 non-null  float64
 13  Power_hp              40494 non-null  int64  
 14  Engine_Size_cc        34564 non-null  float64
 15  Cylinders             30332 no

### Primer contacto
40.494 registros y 25 columnas. Hay bastantes missings en varias columnas: `Fuel_Consumption_l` es la más grave con solo 12.773 valores válidos, pero también `Drivetrain`, `Gears`, `Cylinders` y `Upholstery` tienen huecos considerables. `Year` y `Country` también fallan en algunos registros. El target `Price` está completo, que es lo importante.

`Image_url` son links a fotos de los anuncios en AutoScout24, no aportan nada al análisis y parecen al menos muchos ya caducados (404)

In [75]:
df.drop(columns=["Image_url"], inplace=True)  # URLs caducadas, no sirven
df.shape


(40494, 24)

In [76]:
df.describe()

,Mileage_km,Price,Year,Fuel_Consumption_l,Gears,Power_hp,Engine_Size_cc,Cylinders,Seats,Doors,Previous_Owners
count,4.049400e+04,4.049400e+04,38799.000000,12773.000000,25115.000000,40494.000000,34564.000000,30332.000000,38697.000000,39517.000000,22271.000000
mean,7.723058e+04,5.662527e+04,2017.423877,6.727151,5.899024,231.339976,2275.084597,4.713339,4.633667,4.250879,1.406313
std,3.216313e+05,1.820272e+05,9.141150,3.544475,2.012859,179.923677,1540.972728,2.201816,1.652199,1.113654,1.027425
min,0.000000e+00,1.000000e+00,1922.000000,0.100000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000,0.000000
25%,1.700000e+04,1.368350e+04,2015.000000,4.700000,5.000000,116.000000,1353.000000,4.000000,5.000000,4.000000,1.000000
50%,5.599500e+04,2.219950e+04,2020.000000,5.600000,6.000000,158.000000,1598.000000,4.000000,5.000000,5.000000,1.000000
75%,1.059982e+05,3.600000e+04,2023.000000,7.200000,7.000000,280.000000,2494.000000,4.000000,5.000000,5.000000,2.000000
max,1.677722e+07,9.999999e+06,2026.000000,55.000000,69.000000,3399.000000,67500.000000,50.000000,255.000000,25.000000,94.000000


In [77]:
df["Country"].value_counts(dropna=False) #Vemos países y cuántos no tienen país

Country
DE     15494
IT      9769
NL      4179
BE      3789
NaN     2592
ES      1857
FR      1363
AT      1338
LU       113
Name: count, dtype: int64

## Paso 3: Hipótesis

Antes de entrar en el análisis planteamos las hipótesis que guiarán el EDA. Partimos de 6 candidatas — algunas se confirmarán, otras se descartarán o fusionarán según lo que vayan mostrando los datos.

1. **Kilometraje vs antigüedad:** ¿Cuál de los dos factores deprecia más el precio?
2. **Diferencias por país:** ¿Hay diferencias significativas de precio entre países para características similares?
3. **Retención de valor por marca:** ¿Hay marcas que deprecian significativamente menos que otras?
4. **Tipo de combustible:** ¿El combustible afecta al precio de forma significativa?
5. **Propietarios anteriores:** ¿Un mayor número de propietarios deprecia el precio de forma notable?
6. **Potencia:** ¿Es la potencia el factor técnico con mayor correlación con el precio?

In [78]:
df.dtypes # Tipos

Make                        str
Model                       str
Body                        str
Mileage_km                int64
Price                     int64
Year                    float64
Country                     str
Condition                   str
Fuel_Type                   str
Fuel_Consumption_l      float64
Drivetrain                  str
Gearbox                     str
Gears                   float64
Power_hp                  int64
Engine_Size_cc          float64
Cylinders               float64
Seats                   float64
Doors                   float64
Color                       str
Upholstery                  str
Full_Service_History       bool
Non_Smoker_Vehicle         bool
Previous_Owners         float64
Seller                      str
dtype: object

In [79]:
df.isnull().sum() # Missings

Make                        0
Model                     221
Body                        1
Mileage_km                  0
Price                       0
Year                     1695
Country                  2592
Condition                   0
Fuel_Type                  29
Fuel_Consumption_l      27721
Drivetrain               8634
Gearbox                   480
Gears                   15379
Power_hp                    0
Engine_Size_cc           5930
Cylinders               10162
Seats                    1797
Doors                     977
Color                    2575
Upholstery               7272
Full_Service_History        0
Non_Smoker_Vehicle          0
Previous_Owners         18223
Seller                    150
dtype: int64

## Paso 4: Tabla de variables

Con los tipos y missings analizados, inventariamos todas las columnas y su utilidad esperada.

| Variable | Tipo | Missings | % | Uso previsto |
|----------|------|----------|---|--------------|
| Make | Categórica | 0 | 0% | Usar |
| Model | Categórica | 221 | 0.5% | Usar |
| Body | Categórica | 1 | 0% | Usar |
| Mileage_km | Numérica | 0 | 0% | Usar |
| Price | Numérica | 0 | 0% | Target |
| Year | Numérica | 1.695 | 4.2% | Usar, tratable |
| Country | Categórica | 2.592 | 6.4% | Usar, clave para el análisis |
| Condition | Categórica | 0 | 0% | Usar |
| Fuel_Type | Categórica | 29 | 0.1% | Usar |
| Fuel_Consumption_l | Numérica | 27.721 | 68.5% | Descartar |
| Drivetrain | Categórica | 8.634 | 21.3% | A valorar |
| Gearbox | Categórica | 480 | 1.2% | Usar |
| Gears | Numérica | 15.379 | 38% | Descartar |
| Power_hp | Numérica | 0 | 0% | Usar |
| Engine_Size_cc | Numérica | 5.930 | 14.6% | A valorar |
| Cylinders | Numérica | 10.162 | 25.1% | A valorar |
| Seats | Numérica | 1.797 | 4.4% | Usar, tratable |
| Doors | Numérica | 977 | 2.4% | Usar, tratable |
| Color | Categórica | 2.575 | 6.4% | A valorar, baja relevancia esperada |
| Upholstery | Categórica | 7.272 | 18% | A valorar |
| Full_Service_History | Booleana | 0 | 0% | Usar |
| Non_Smoker_Vehicle | Booleana | 0 | 0% | Usar |
| Previous_Owners | Numérica | 18.223 | 45% | A valorar, relevante pero muy incompleta |
| Seller | Categórica | 150 | 0.4% | Usar |

`Fuel_Consumption_l` y `Gears` se descartan por missings excesivos. El resto marcado como "a valorar" se decidirá durante la limpieza según lo que aporten al análisis.

## Paso 5: Limpieza de datos

Empezamos descartando las columnas con missings excesivos ya identificados.

In [80]:
df.drop(columns=["Fuel_Consumption_l", "Gears"], inplace=True) # Eliminamos columnas
df.shape

(40494, 22)

### Outliers imposibles

El describe inicial reveló valores claramente erróneos en varias columnas. Los investigamos y filtramos.

In [81]:
df["Mileage_km"].describe() #Vemos kms

count    4.049400e+04
mean     7.723058e+04
std      3.216313e+05
min      0.000000e+00
25%      1.700000e+04
50%      5.599500e+04
75%      1.059982e+05
max      1.677722e+07
Name: Mileage_km, dtype: float64

In [82]:
print(f"Mileage > 300.000: {(df['Mileage_km'] > 300_000).sum()}") # Contamos por rangos
print(f"Mileage > 400.000: {(df['Mileage_km'] > 400_000).sum()}")
print(f"Mileage > 500.000: {(df['Mileage_km'] > 500_000).sum()}")
print(f"Mileage == 0: {(df['Mileage_km'] == 0).sum()}")

Mileage > 300.000: 281
Mileage > 400.000: 50
Mileage > 500.000: 25
Mileage == 0: 896


In [83]:
df[df['Mileage_km'] > 400_000][['Make', 'Model', 'Year', 'Mileage_km', 'Price', 'Country']].sort_values('Mileage_km', ascending=False).head(20)
# Muchos km? Fallos claros de NL

,Make,Model,Year,Mileage_km,Price,Country
414,Abarth,595 Turismo,2013.0,16777215,8500,NL
3854,Bentley,Brooklands,1993.0,16777215,5500,NL
7073,Chevrolet,Camaro,1985.0,16777215,6500,NL
3881,Bentley,Continental GT,2004.0,16777215,19500,NL
3956,Bentley,Continental GTC,2008.0,16777215,29500,NL
4089,Bentley,S2,1960.0,16777215,22500,NL
7158,Chevrolet,Camaro,1997.0,16777215,4500,NL
22345,Lexus,NaN,2023.0,16777215,29500,NL
20640,Lancia,Delta,1988.0,16777215,22500,NL
19612,Lamborghini,NaN,1967.0,16777215,17500,NL


In [84]:
df["Price"].describe() #vemos precios y luego los rangos

count    4.049400e+04
mean     5.662527e+04
std      1.820272e+05
min      1.000000e+00
25%      1.368350e+04
50%      2.219950e+04
75%      3.600000e+04
max      9.999999e+06
Name: Price, dtype: float64

In [85]:
print(f"Price < 500: {(df['Price'] < 500).sum()}")
print(f"Price > 200.000: {(df['Price'] > 200_000).sum()}")
print(f"Price > 300.000: {(df['Price'] > 300_000).sum()}")
print(f"Price > 500.000: {(df['Price'] > 500_000).sum()}")

Price < 500: 27
Price > 200.000: 2569
Price > 300.000: 1445
Price > 500.000: 470


In [86]:
df[df['Price'] > 200_000][['Make', 'Model', 'Year', 'Mileage_km', 'Price', 'Country']].sort_values('Price', ascending=False).head(20)
#muchos coches caros, vamos a verlos
# Citroen C3 de oro, brillantes y rubíes

,Make,Model,Year,Mileage_km,Price,Country
8280,Citroen,C3,2009.0,85516,9999999,BE
12156,Ferrari,Enzo Ferrari,2004.0,14900,7500000,DE
12203,Ferrari,F50,1996.0,32000,6500000,FR
12464,Ferrari,288,1985.0,1,6500000,ES
12064,Ferrari,Enzo Ferrari,2003.0,13222,6400000,FR
12060,Ferrari,Daytona,NaN,0,5850000,FR
12222,Ferrari,LaFerrari,2018.0,900,5700000,ES
12063,Ferrari,F50,1996.0,20666,5600000,FR
12180,Ferrari,NaN,NaN,50,5600000,FR
12334,Ferrari,Enzo Ferrari,2004.0,3988,5400000,FR


In [87]:
df[df['Price'] < 500][['Make', 'Model', 'Year', 'Mileage_km', 'Price', 'Country']]
#Veamos los muy baratos

,Make,Model,Year,Mileage_km,Price,Country
6702,Chevrolet,Kalos,2008.0,66638,499,DE
6714,Chevrolet,Lacetti,2005.0,156350,450,DE
6818,Chevrolet,Aveo,2009.0,327951,469,DE
6835,Chevrolet,Lacetti,2009.0,106900,350,DE
6884,Chevrolet,Aveo,2010.0,131112,399,DE
6893,Chevrolet,Matiz,2005.0,84000,300,DE
6906,Chevrolet,Matiz,2009.0,91000,250,DE
6920,Chevrolet,Matiz,2009.0,74000,400,DE
6962,Chevrolet,Matiz,2007.0,73821,400,NaN
6966,Chevrolet,Kalos,2006.0,173282,300,NaN


In [88]:
print(f"Price == 1: {(df['Price'] == 1).sum()}") #Menos de 100? yo quiero
print(f"Price < 100: {(df['Price'] < 100).sum()}")

Price == 1: 2
Price < 100: 3


In [89]:
df = df[(df['Price'] >= 100) & (df['Price'] < 9_999_999)] #quitamos exremos

In [90]:
print(f"Power_hp == 0: {(df['Power_hp'] == 0).sum()}")   #Vamos a revisar los cv
df[df['Power_hp'] == 0][['Make', 'Model', 'Year', 'Power_hp', 'Price', 'Country']].head(10)  

Power_hp == 0: 419


,Make,Model,Year,Power_hp,Price,Country
138,Abarth,595,2021.0,0,17490,IT
149,Abarth,595,2021.0,0,17490,IT
155,Abarth,595,2021.0,0,17490,IT
170,Abarth,500C,2010.0,0,6990,FR
172,Abarth,595,2019.0,0,12890,FR
414,Abarth,595 Turismo,2013.0,0,8500,NL
737,Abarth,595,2018.0,0,13990,FR
1160,Alfa Romeo,Tonale,2025.0,0,44900,IT
1321,Alfa Romeo,Spider,1998.0,0,6990,ES
1510,Alfa Romeo,Tonale,2025.0,0,44400,IT


In [91]:
df = df[df['Power_hp'] > 0]  #Me gustaría que hubiera menos, porque hay que eliminar, son errores

In [92]:
df = df[df['Mileage_km'] <= 500_000] #Los kms

In [93]:
df.shape  #Razonable, hemos eliminado pocos

(40060, 22)

In [94]:
print(f"Year < 1980: {(df['Year'] < 1980).sum()}")  # Vemos años
print(f"Year > 2026: {(df['Year'] > 2026).sum()}")
df['Year'].value_counts().sort_index().head(20)

Year < 1980: 485
Year > 2026: 0


Year
1926.0     1
1929.0     2
1934.0     4
1935.0     1
1936.0     3
1937.0     2
1949.0     3
1950.0     1
1952.0     2
1953.0     2
1954.0     7
1955.0     6
1956.0     6
1957.0    13
1958.0     6
1959.0    14
1960.0    13
1961.0    12
1962.0    12
1963.0    15
Name: count, dtype: int64

In [95]:
df[df['Year'] < 1970][['Make', 'Model', 'Year', 'Mileage_km', 'Price', 'Country']].sort_values('Year').head(30)
# Más de cerca los coches antiguos por si son errores

,Make,Model,Year,Mileage_km,Price,Country
33363,Rolls-Royce,NaN,1926.0,32049,109000,AT
33174,Rolls-Royce,Phantom,1929.0,196976,99900,NL
33312,Rolls-Royce,Phantom,1929.0,71997,63900,NL
33077,Rolls-Royce,NaN,1934.0,69000,43900,DE
33222,Rolls-Royce,NaN,1934.0,48000,95000,NaN
33386,Rolls-Royce,NaN,1934.0,1000,95000,NaN
33366,Rolls-Royce,NaN,1934.0,385,48000,IT
32841,Rolls-Royce,Park Ward,1935.0,80000,31500,NaN
32821,Rolls-Royce,NaN,1936.0,8320,92500,DE
33331,Rolls-Royce,NaN,1936.0,155300,109900,FR


In [96]:
print(f"Engine_Size_cc > 10.000: {(df['Engine_Size_cc'] > 10_000).sum()}")
df[df['Engine_Size_cc'] > 10_000][['Make', 'Model', 'Year', 'Engine_Size_cc', 'Price', 'Country']].head(10)
#Cilindradas por encima de 8.000 ya es bien raro

Engine_Size_cc > 10.000: 3


,Make,Model,Year,Engine_Size_cc,Price,Country
631,Abarth,695,2010.0,14368.0,26990,DE
11303,Dodge,Charger,2010.0,56654.0,16699,DE
32766,Rolls-Royce,Silver Spirit,1978.0,67500.0,19900,DE


In [97]:
#df = df[df['Engine_Size_cc'] <= 10_000]
#df.shape
#Limpiamos
#Me he cargao los nulos....
df = df[(df['Engine_Size_cc'] <= 10_000) | (df['Engine_Size_cc'].isna())]
df.shape

(40057, 22)

In [98]:
# Ahora sí, 40057 razonable

In [99]:
df = df[(df['Cylinders'] <= 12) | (df['Cylinders'].isna())]
df.shape
#Más de 12 cilindros ni un F1

(40056, 22)

In [100]:
# Asientos
df[df['Seats'] > 9][['Make', 'Model', 'Year', 'Seats', 'Price', 'Country']].head(20)

,Make,Model,Year,Seats,Price,Country
11093,Dodge,Challenger,2019.0,44.0,36990,DE
27881,Mitsubishi,Outlander,2014.0,255.0,10990,FR


In [102]:
df = df[(df['Seats'] <= 9) | (df['Seats'].isna())] #Limpiamos más de 9
df.shape

(40054, 22)

In [103]:
df['Doors'].value_counts().sort_index() #Vemos puertas

Doors
1.0        11
2.0      5850
3.0      2561
4.0      6483
5.0     24209
6.0        39
25.0        1
Name: count, dtype: int64

In [105]:
df = df[(df['Doors'] <= 6) | (df['Doors'].isna())] #Limpieza más de 6
df.shape

(40053, 22)

### Resumen de limpieza

- Columnas eliminadas: `Image_url` (URLs caducadas), `Fuel_Consumption_l` (68.5% missings), `Gears` (38% missings)
- Precios imposibles eliminados: valores por debajo de 100€ y el valor 9.999.999€ (código de error)
- Potencia 0 cv: 419 registros eliminados
- Kilometraje superior a 500.000 km eliminado
- Engine_Size_cc superior a 10.000 cc: 3 registros eliminados
- Cylinders superior a 12: 1 registro eliminado
- Seats superior a 9: 2 registros eliminado
- Doors superior a 6: 1 registro eliminado

Dataset resultante: 40.053 registros y 22 columnas. Muy razonable.

In [107]:
#Missings
df.isnull().sum().sort_values(ascending=False)

Previous_Owners         17890
Cylinders                9837
Drivetrain               8347
Upholstery               7042
Engine_Size_cc           5612
Country                  2582
Color                    2531
Year                     1682
Seats                    1562
Doors                     900
Gearbox                   455
Model                     211
Seller                    145
Fuel_Type                  24
Body                        1
Price                       0
Make                        0
Mileage_km                  0
Power_hp                    0
Condition                   0
Non_Smoker_Vehicle          0
Full_Service_History        0
dtype: int64

In [108]:
print(df['Drivetrain'].value_counts())
print(df['Upholstery'].value_counts())

Drivetrain
Front Wheel Drive    17338
4WD                   8903
Rear Wheel Drive      5465
Name: count, dtype: int64
Upholstery
Cloth           12880
Full leather    12223
Part leather     3918
Other            2076
alcantara        1703
Velour            211
Name: count, dtype: int64


In [110]:
df['Drivetrain'] = df['Drivetrain'].fillna('Front Wheel Drive')
df['Upholstery'] = df['Upholstery'].fillna('Cloth')
df['Engine_Size_cc'] = df['Engine_Size_cc'].fillna(df['Engine_Size_cc'].median())

### Tratamiento de missings

**Columnas eliminadas por missings excesivos o baja utilidad:**
- `Previous_Owners` — 44% de missings y dato fácilmente manipulable por los vendedores. Descartamos también la hipótesis 5.
- `Cylinders` — 24% de missings y redundante teniendo `Power_hp`. La potencia es lo que mira el comprador, no el número de cilindros.

**Columnas imputadas:**
- `Drivetrain` — imputado con "Front Wheel Drive". En los anuncios se destaca siempre si el coche tiene tracción trasera o 4x4, por lo que el missing casi seguro indica tracción delantera, la más común del mercado europeo.
- `Upholstery` — imputado con "Cloth". Mismo razonamiento: el cuero se anuncia, el silencio indica tapicería básica.
- `Engine_Size_cc` — imputado con la mediana. Menos relevante que la potencia pero no hay que perder filas innecesariamente.

In [111]:
df.isnull().sum().sort_values(ascending=False)

Previous_Owners         17890
Cylinders                9837
Country                  2582
Color                    2531
Year                     1682
Seats                    1562
Doors                     900
Gearbox                   455
Model                     211
Seller                    145
Fuel_Type                  24
Body                        1
Make                        0
Mileage_km                  0
Price                       0
Power_hp                    0
Condition                   0
Drivetrain                  0
Engine_Size_cc              0
Upholstery                  0
Non_Smoker_Vehicle          0
Full_Service_History        0
dtype: int64

In [112]:
df.drop(columns=['Previous_Owners', 'Cylinders'], inplace=True)
df.isnull().sum().sort_values(ascending=False)

Country                 2582
Color                   2531
Year                    1682
Seats                   1562
Doors                    900
Gearbox                  455
Model                    211
Seller                   145
Fuel_Type                 24
Body                       1
Mileage_km                 0
Make                       0
Power_hp                   0
Drivetrain                 0
Price                      0
Condition                  0
Engine_Size_cc             0
Upholstery                 0
Full_Service_History       0
Non_Smoker_Vehicle         0
dtype: int64

In [115]:
# Vamos a ver los años a ver si son coches raros o hay de todo
df[df['Year'].isna()][['Make', 'Model', 'Mileage_km', 'Price', 'Country']].head(20)

,Make,Model,Mileage_km,Price,Country
70,Abarth,600e,1,23400,IT
86,Abarth,600e,10,29750,IT
109,Abarth,600e,0,26900,IT
147,Abarth,600e,0,30000,IT
153,Abarth,600e,10,29950,IT
161,Abarth,600e,0,42950,IT
197,Abarth,500e,50,42490,DE
582,Abarth,600e,0,47490,DE
931,Alfa Romeo,Junior,0,24400,IT
1072,Alfa Romeo,Junior,0,25900,IT


In [116]:
# No me sirve, vamos a ver aleatorios
df[df['Year'].isna()][['Make', 'Model', 'Mileage_km', 'Price', 'Country']].sample(50, random_state=42)

,Make,Model,Mileage_km,Price,Country
31052,Porsche,911,0,240284,DE
11124,Dodge,RAM,10,66999,DE
36615,Suzuki,S-Cross,0,24190,IT
2305,Aston Martin,DB12,30,189007,DE
5697,BYD,Dolphin,0,20990,DE
8789,Corvette,C8,0,133000,IT
15337,Hyundai,i20,0,18799,DE
10149,Dacia,Spring,10,14955,DE
38083,Toyota,Aygo X,1,21233,DE
28544,Nissan,X-Trail,0,38685,DE


In [118]:
# Todo tipo de marcas, no tienen kms y sin fecha. Son km0 sin matricular
df['Year'] = df['Year'].fillna(df['Year'].median()) #Mediana para no perderlos

In [ ]:
df['Gearbox'].value_counts() 

Gearbox
Automatic         25053
Manual            13816
Semi-automatic      729
Name: count, dtype: int64

In [120]:
# Buscamos moda y sorpresa en Las Gaunas, muchos automáticos, no puedo imputar con la moda y sól "sólo" 455 así que eliminamos
df = df.dropna(subset=['Gearbox'])
df.shape

(39598, 20)

In [121]:
# Eliminamos Filas de los que tienen pocos missings
df = df.dropna(subset=['Model', 'Seller', 'Fuel_Type', 'Body'])
df.shape

(39247, 20)

In [123]:
 # Asientos y número de puertas sí podemos ponerle la moda a los missings
df['Seats'] = df['Seats'].fillna(df['Seats'].mode()[0])
df['Doors'] = df['Doors'].fillna(df['Doors'].mode()[0])

In [124]:
# Repaso
df.isnull().sum().sort_values(ascending=False)

Color                   2350
Country                 2173
Make                       0
Model                      0
Mileage_km                 0
Body                       0
Year                       0
Price                      0
Fuel_Type                  0
Drivetrain                 0
Gearbox                    0
Condition                  0
Power_hp                   0
Engine_Size_cc             0
Seats                      0
Doors                      0
Upholstery                 0
Full_Service_History       0
Non_Smoker_Vehicle         0
Seller                     0
dtype: int64

In [125]:
# Color interesante para estudiar otro día, no lo metemos en hipótesis
df.drop(columns=['Color'], inplace=True)
df.shape

(39247, 19)

### Decisiones sobre missings restantes

- `Color` — eliminada la columna. Aunque una estadística de colores sería curiosa, no aporta valor predictivo al modelo y tiene 2.350 missings. Preferimos no introducir ruido.
- `Year` — imputado con la mediana. La muestra de registros sin año incluía coches nuevos sin matricular (0 km, stock de concesionario) sin un patrón sospechoso, por lo que la mediana es una imputación razonable.
- `Seats` y `Doors` — imputados con la moda. Valores muy estables en el mercado general.
- `Gearbox` — eliminadas las 455 filas. Sorprendentemente la moda es "Automatic", contra la intuición del mercado europeo tradicional. Preferimos no imputar un dato tan relevante y eliminar las filas, que son pocas.
- `Model`, `Seller`, `Fuel_Type`, `Body` — eliminadas las filas directamente por ser pocos registros.
- `Country` — mantenemos los missings de momento. Se eliminarán al filtrar por España para el modelo.

Dataset resultante: 39.247 registros y 19 columnas.